# Lecture 3 (Notebook 2 / 2) — Propagation & Light: **PROPOSAL** + **Prometheus**

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

This is the **second of two** notebooks for the Lecture 3 hands-on. The first
(`1_SIREN_flux_injection_weights.ipynb`) walked the **flux → interaction** stages with
**SIREN**: it sampled an atmospheric/astrophysical flux, injected $\nu$ interactions in ice,
and weighted them. We pick up that $\nu$ event and follow the rest of the telescope pipeline:

```
  FLUX  →  INTERACTION  →  PROPAGATION  →  LIGHT  →  DETECTOR
  [ ........ SIREN (notebook 1) ....... ]  [ PROPOSAL ]  [ ....... Prometheus ....... ]
                                              (Part 1)        (Parts 2 & 3, this notebook)
```

**What we do here**

- **Part 1 — PROPOSAL (runs live, cheap):** propagate a charged lepton through ice and watch
  the **race between flying and decaying**. We propagate $\tau$ leptons across a log-energy
  range and compare the *real* decay-length distribution to the naive $L=\gamma c\tau$ curve —
  this is the physics behind the $\nu_\tau$ **double-bang**.
- **Part 2 — Prometheus (install + tiny live run, then load cached):** Prometheus glues
  injection → PROPOSAL → Cherenkov light → detector together. We run a *minimal* simulation
  live just to prove the install works in Colab, then load **pre-generated IceCube_HE** events
  for the polished displays.
- **Part 3 — the signature zoo:** render all four IceCube_HE topologies — $\nu_e$ **cascade**,
  $\nu_\mu$ **track**, $\nu_\tau$ **double-bang**, and a $\nu_\mu$ event — the *"full zoo of
  event signatures"* from Lecture 2, and discuss energy vs. direction reconstruction.


## Setup

On Colab the GitHub browser loads **only this `.ipynb`** into a fresh VM, so `src/helpers.py`
and `data/` are absent. The cell below clones the repo (set `REPO_URL`), puts `src/` on the
path, and installs **Prometheus** via its **built-in installer** — which *also builds PROPOSAL*
and LeptonInjector (see the
[built-in installer docs](https://harvard-neutrino.github.io/prometheus/installation/built-in-installer/)).

> **Heads-up (Colab timing).** The Prometheus installer **compiles C++ from source** (PROPOSAL,
> LeptonInjector, optionally PPC) and can take **10–20 min**. It is set to run only on Colab. If
> you just want the event displays, you can skip the build: every display in Part 3 reads the
> **pre-generated parquet** in `data/`, so it works even if the live install hasn't finished.


In [ ]:
# === Setup: clone the repo (Colab), put src/ on the path, install Prometheus+PROPOSAL ===
import os, sys, subprocess

REPO_URL = 'https://github.com/USER/REPO.git'   # <-- set to your repository
SUBDIR   = 'Lecture3_Simulation'                # folder that holds src/ and data/
ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB and not os.path.isdir('REPO'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'REPO'], check=True)

# Find the folder containing src/helpers.py, whether cloned (Colab) or in-repo (local):
_cands = ['REPO/' + SUBDIR, SUBDIR, '.', '..', '../..']
_base  = next((p for p in _cands if os.path.isdir(os.path.join(p, 'src'))), None)
assert _base, 'Could not find src/ -- check REPO_URL / SUBDIR.'
sys.path.insert(0, os.path.abspath(os.path.join(_base, 'src')))

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

if ON_COLAB:
    pip('pyarrow', 'awkward')                    # light deps: read the cached parquet
    # Prometheus' built-in installer -- it also builds PROPOSAL + LeptonInjector.
    # NOTE: compiles from source (~10-20 min). '--with-ppc' adds the ice photon path.
    if not os.path.isdir('prometheus'):
        subprocess.run(['git', 'clone',
                        'https://github.com/Harvard-Neutrino/prometheus.git'], check=True)
        # Build PROPOSAL + LeptonInjector (and PPC for ice):
        subprocess.run('cd prometheus && bash install.sh --with-ppc',
                       shell=True, check=True)
        # The installer puts the package under ./prometheus; add it to the path:
        sys.path.insert(0, os.path.abspath('prometheus'))

import numpy as np, matplotlib.pyplot as plt
import helpers as H
print('helpers loaded from:', H.__file__)
for _m in ('proposal', 'prometheus'):
    print(f'  {_m:11s} importable: {H.have(_m)}')


---
## Part 1 — Lepton propagation with **PROPOSAL**

A $\nu_\tau$ charged-current interaction makes a $\tau$ at the vertex (the first "bang"). The
$\tau$ then **flies**, losing energy, and eventually **decays** (the second "bang"). Whether a
detector resolves the two cascades depends on the **decay length**

$$ L = \gamma\,c\tau \;=\; \frac{E_\tau}{m_\tau}\,c\tau \;\approx\; 50\ \mathrm{m}\times\frac{E_\tau}{\mathrm{PeV}} $$

(*Lecture 2, tau page:* $d_\tau \approx 50\,\mathrm{m}\cdot(E_\tau/\mathrm{PeV})$, the
**decay-vs-radiative** competition). Too low in energy → the two bangs merge into one cascade;
too high → the $\tau$ leaves the detector before decaying (a track). The "double-bang" lives in
a **resolvable window** set by the module spacing (~125 m in IceCube).

This is the same propagation physics as the **muon** and **electron**, run by **PROPOSAL**
(Lecture 2: Bethe-Bloch continuous loss + **stochastic** bremsstrahlung / pair / photonuclear
losses, the **critical energy** where radiative loss overtakes ionization, and EM vs. hadronic
showers). The naive $\gamma c\tau$ formula ignores these losses; PROPOSAL includes them, so the
*real* decay-length distribution bends away from the naive curve at the highest energies.


In [ ]:
# Reference curve: naive mean decay length L = (E/m_tau)*c*tau (NO energy loss).
E_tau = np.logspace(4, 7, 100)                 # 10 TeV - 10 PeV
L_mean = H.tau_mean_decay_length_m(E_tau)

plt.figure(figsize=(7,5))
plt.loglog(E_tau, L_mean, label=r'naive $\gamma c\tau$ (no losses)')
plt.axhspan(10, 1000, alpha=.12, color='C2', label='resolvable window (~10-1000 m)')
plt.axhline(125, ls='--', color='grey')
plt.text(1.2e4, 140, 'IceCube string spacing ~125 m')
plt.xlabel(r'$E_\tau$ [GeV]'); plt.ylabel('tau decay length [m]')
plt.legend(); plt.title('Why double-bangs live around a PeV')
plt.grid(alpha=.3, which='both'); plt.show()


### Propagate real $\tau$ leptons with PROPOSAL

We build a minimal PROPOSAL propagator for the $\tau$ in **homogeneous ice** and propagate many
taus at each energy, recording the distance at which each one decays. This is *exactly* the API
Prometheus uses internally (`prometheus/lepton_propagation/new_proposal_lepton_propagator.py`),
reduced to a single medium so it runs in seconds.

PROPOSAL works in **MeV** and **cm**. Key calls (PROPOSAL ≥ 7):
`pp.particle.TauMinusDef()`, `pp.EnergyCutSettings`, `pp.crosssection.make_std_crosssection`,
`pp.PropagationUtility`, `pp.Propagator(pdef, [(geometry, utility, density)])`,
`prop.propagate(state, max_distance_cm)`, then `.track_propagated_distances()` for the decay
point and `.decay_products()` for the daughters.


In [ ]:
def build_tau_propagator():
    'Minimal PROPOSAL tau propagator in homogeneous ice (one big sphere).'
    import proposal as pp
    GeV = 1e3                       # MeV per GeV
    pdef = pp.particle.TauMinusDef()
    medium = pp.medium.Ice()
    cuts = pp.EnergyCutSettings(0.5 * GeV, 1.0, True)   # ecut=0.5 GeV, vcut=1, cont.rand.
    cross = pp.crosssection.make_std_crosssection(pdef, medium, cuts, True)
    collection = pp.PropagationUtilityCollection()
    collection.displacement = pp.make_displacement(cross, True)
    collection.interaction  = pp.make_interaction(cross, True)
    collection.time         = pp.make_time(cross, pdef, True)
    collection.decay        = pp.make_decay(cross, pdef, True)   # << lets the tau DECAY
    utility = pp.PropagationUtility(collection=collection)
    geo = pp.geometry.Sphere(pp.Cartesian3D(0, 0, 0), 1e8, 0)   # 1000 km radius, in cm
    density = pp.density_distribution.density_homogeneous(1.0)   # use medium's own density
    return pp.Propagator(pdef, [(geo, utility, density)])

def propagate_taus(prop, energy_gev, n=300, max_m=1e5):
    'Return decay lengths [m] for n taus of given energy.'
    import proposal as pp
    GeV, m_to_cm, cm_to_m = 1e3, 100.0, 0.01
    out = []
    for _ in range(n):
        st = pp.particle.ParticleState()
        st.position  = pp.Cartesian3D(0, 0, 0)
        st.direction = pp.Cartesian3D(0, 0, 1)
        st.energy    = energy_gev * GeV
        sec = prop.propagate(st, max_m * m_to_cm)
        out.append(sec.track_propagated_distances()[-1] * cm_to_m)
    return np.array(out)

if H.have('proposal'):
    prop = build_tau_propagator()
    energies = np.array([1e5, 1e6, 1e7])        # 0.1, 1, 10 PeV
    decays = {E: propagate_taus(prop, E, n=300) for E in energies}
    print('propagated', sum(len(v) for v in decays.values()), 'taus')
else:
    decays = None
    print('proposal not installed -- run the Setup cell on Colab; '
          'the displays in Parts 2-3 still work from cached data.')


In [ ]:
# Decay-length distribution per energy band + median vs the naive gamma*c*tau curve.
if decays is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = {1e5:'0.1 PeV', 1e6:'1 PeV', 1e7:'10 PeV'}
    for E, L in decays.items():
        ax[0].hist(L, bins=np.logspace(-1, 4, 50), histtype='step',
                   density=True, label=labels[E])
    ax[0].axvspan(10, 1000, alpha=.12, color='C2')
    ax[0].set_xscale('log'); ax[0].set_xlabel('decay length [m]')
    ax[0].set_ylabel('pdf'); ax[0].legend(title=r'$E_\tau$')
    ax[0].set_title('PROPOSAL tau decay lengths')

    Es = np.array(sorted(decays))
    med = np.array([np.median(decays[E]) for E in Es])
    ax[1].loglog(Es, med, 'o-', label='PROPOSAL median')
    ax[1].loglog(E_tau, L_mean, 'k--', label=r'naive $\gamma c\tau$ (no losses)')
    ax[1].axhspan(10, 1000, alpha=.12, color='C2', label='resolvable window')
    ax[1].set_xlabel(r'$E_\tau$ [GeV]'); ax[1].set_ylabel('median decay length [m]')
    ax[1].legend(); ax[1].set_title('Real vs naive decay length')
    plt.tight_layout(); plt.show()
else:
    print('(skipped -- needs PROPOSAL; see Setup)')


> **★ Discussion (Lecture 2 callback).** The naive $\gamma c\tau$ curve and PROPOSAL agree at
> low energy but the PROPOSAL median falls *below* it at the highest energies — the $\tau$'s
> stochastic energy losses bleed off energy (lowering $\gamma$) *before* it decays. Where does
> the median decay length cross the ~125 m string spacing? That energy is roughly where
> double-bangs become resolvable. Swap `TauMinusDef()` → `MuMinusDef()` and remove the decay
> term: the muon now **ranges out** instead of decaying — that is the "track" topology you'll
> see in Part 3.


---
## Part 2 — **Prometheus**: the full light + detector simulation

**Prometheus** is the open-source neutrino-telescope simulation. It chains the whole pipeline:

```
 injection (LeptonInjector/SIREN)  →  lepton propagation (PROPOSAL)
        →  Cherenkov light yield  →  photon propagation to modules  →  detected photons
```

The photon step has two backends: **PPC** (GPU, the IceCube-style path; `--with-ppc`) and
**olympus** (a CPU parametric path, slower per photon but GPU-free). The pre-generated
IceCube_HE data below was made with the **PPC** path on a GPU; for a quick Colab check we run a
*tiny* `olympus` CPU job on a small demo detector.

The config is a nested dict (`prometheus.config`). The keys we set match the real defaults in
`prometheus/config.py`: `run.nevents`, `injection.name`, `detector.geo file`,
`photon propagator.name`. You build `Prometheus()` and call `.sim()`.


In [ ]:
# Show the Prometheus config skeleton (the keys that connect the pipeline pieces).
if H.have('prometheus'):
    from prometheus import config
    print('injection backends   :', list(config['injection'].keys()))
    print('lepton propagator    :', config['lepton propagator']['name'],
          '(PROPOSAL under the hood)')
    print('photon propagators   :', [k for k in config['photon propagator']
                                     if isinstance(config['photon propagator'][k], dict)])
    print('default nevents      :', config['run']['nevents'])
else:
    print('prometheus not importable here -- this cell just inspects its config dict.')


In [ ]:
# ---- TINY live Prometheus run to verify the Colab install (CPU 'olympus' path) ----
# This is a smoke test, NOT the polished display: a handful of events on the small
# demo water detector with the CPU photon path. The real displays load cached data below.
if H.have('prometheus'):
    import glob, os
    from prometheus import Prometheus, config

    # locate the bundled resources of the installed/cloned prometheus
    res = next((p for p in ['prometheus/resources', 'REPO/prometheus/resources',
                            'resources'] if os.path.isdir(p)), None)
    config['run']['nevents'] = 2                       # tiny!
    config['run']['storage prefix'] = './prom_smoke/'
    os.makedirs('./prom_smoke', exist_ok=True)
    config['injection']['name'] = 'LeptonInjector'
    config['injection']['LeptonInjector']['simulation']['minimal energy'] = 1e4
    config['injection']['LeptonInjector']['simulation']['maximal energy'] = 1e5
    config['photon propagator']['name'] = 'olympus'    # CPU path, no GPU needed
    if res:
        config['detector']['geo file'] = f'{res}/geofiles/demo_water.geo'
    try:
        p = Prometheus()
        p.sim()
        out = glob.glob('./prom_smoke/*photons.parquet')
        print('Prometheus smoke test OK -> wrote', out)
    except Exception as e:
        print('Smoke test could not complete here:', type(e).__name__, e)
        print('(That is fine for the displays -- they use the cached IceCube_HE data.)')
else:
    print('prometheus not installed -- skipping live smoke test; '
          'the displays below use the pre-generated IceCube_HE data.')


### Load a pre-generated IceCube_HE event

The committed `data/Prometheus_simulation/IceCube_HE/<signature>/Generation_*_photons.parquet`
files are **real Prometheus output**. Each file is an `awkward` array, one record per event:

- `mc_truth` — the injected truth (`initial_state_type`/`energy`, `final_state_*`,
  `interaction`, `bjorken_x/y`, ...).
- `photons` — **one entry per detected photon**: `sensor_pos_x/y/z` (the module it hit),
  `string_id`, `sensor_id`, `t` (arrival time, ns).

`helpers.load_prometheus_event` reads this schema, collapses photons to **one dot per module**
(charge = photon count, time = earliest arrival), and loads the full detector geometry from the
`.geo` file named in the parquet's `config_prometheus` metadata for the faint outline.


In [ ]:
TAU = 'data/Prometheus_simulation/IceCube_HE/TauMinus/Generation_00000-000_photons.parquet'
hits, geo, info = H.load_prometheus_event(TAU, event='brightest')
print('event', info['event_index'], ':', info['initial_state_label'],
      f"E = {info['initial_state_energy_gev']:.2e} GeV")
print(info['n_photons'], 'photons on', info['n_modules_hit'], 'modules')
print('geometry modules for the outline:', None if geo is None else len(geo))


In [ ]:
ax = H.plot_event_display(
    hits, geo=geo,
    title=r'IceCube_HE  $\nu_\tau$ CC  (time-coloured photon hits)')
plt.show()


---
## Part 3 — The signature zoo (Lecture 2 callback)

Same machinery, different final states → different **topologies**. With truth-labelled
Prometheus output we render each one and read off how well each reconstructs energy vs direction:

| Signature | Origin | Topology | Energy reco | Direction reco |
|---|---|---|---|---|
| **Cascade** | $\nu_e$ CC, all NC | compact blob | **good** (calorimetric, contained) | poor |
| **Track** | $\nu_\mu$ CC (muon exits) | long line | poor (only $dE/dx$ lower bound) | **good** (lever arm) |
| **Double bang** | $\nu_\tau$ CC | two blobs | good if both contained | medium |

Below: the four IceCube_HE signatures side by side. Watch the **time colour** (early = red): a
track shows a clear early→late gradient along the muon's path; a cascade lights up almost at once.


In [ ]:
SIGS = [
    ('EMinus',   r'$\nu_e$ CC  →  cascade'),
    ('MuMinus',  r'$\nu_\mu$ CC  →  track'),
    ('TauMinus', r'$\nu_\tau$ CC  →  double-bang'),
    ('NuMu',     r'$\nu_\mu$ event'),
]
fig = plt.figure(figsize=(14, 11))
for i, (sig, title) in enumerate(SIGS):
    path = f'data/Prometheus_simulation/IceCube_HE/{sig}/Generation_00000-000_photons.parquet'
    hits, geo, info = H.load_prometheus_event(path, event='brightest')
    ax = fig.add_subplot(2, 2, i + 1, projection='3d')
    H.plot_event_display(hits, ax=ax, geo=geo,
                         title=f"{title}\n({info['n_modules_hit']} modules, "
                               f"E={info['initial_state_energy_gev']:.1e} GeV)")
plt.tight_layout(); plt.show()


In [ ]:
# Same TauMinus event coloured by CHARGE (NPE) instead of time -- shows where the
# energy is deposited (the two cascades of the double-bang light up brightest).
hits, geo, info = H.load_prometheus_event(TAU, event='brightest')
ax = H.plot_event_display(hits, geo=geo, color_by='npe',
                          title=r'IceCube_HE $\nu_\tau$  (coloured by photon count)')
plt.show()


## ★ Discussion questions & exercises

1. **Decay-length crossover (Part 1).** At what $E_\tau$ does the PROPOSAL median decay length
   cross the ~125 m IceCube string spacing? Relate that to where real IceCube tau searches live
   (~PeV). Why does raising the resolvable-window lower edge (10 m → 30 m, i.e. worse vertex
   resolution) shrink the double-bang energy range?

2. **Stochastic losses (Lecture 2).** Re-run `propagate_taus` at $E_\tau = 10$ PeV with many
   taus and overlay the distribution against the single naive $\gamma c\tau$ value. The spread
   *is* the stochasticity of bremsstrahlung/pair/photonuclear losses. Now switch to
   `MuMinusDef()` (drop `collection.decay`): the muon **ranges out** — measure its range vs
   energy and compare to the muon **critical energy** (~0.5 TeV in ice).

3. **Cascade vs. track reconstruction (Part 3).** From the displays, argue why a contained
   **cascade** gives the best *energy* resolution but poor *direction*, while a **through-going
   track** gives the best *direction* but only a *lower bound* on energy. Which signature is best
   for a diffuse astrophysical flux measurement? For point-source searches?

4. **Connect the notebooks.** The $\nu_\tau$ injected by SIREN in notebook 1 → the decay length
   from PROPOSAL here → the double-bang display. Predict a tau's topology from its energy
   *before* plotting it: low-E taus give a merged single cascade (looks like a $\nu_e$), high-E
   ones two resolved bangs.

5. **Pipeline.** Identify which Prometheus config key controls each pipeline stage
   (`injection.name`, `lepton propagator.name`, `photon propagator.name`,
   `detector.geo file`). Which would you change to (a) simulate ARCA instead of IceCube,
   (b) switch the photon path from GPU PPC to CPU olympus?

---
**References:** Prometheus (arXiv:2304.14526), SIREN / LeptonInjector (arXiv:2012.10449),
PROPOSAL (Koehne et al., Comput. Phys. Commun. 2013; Dunsch et al. 2019),
Formaggio & Zeller (Rev. Mod. Phys. 2012).
